# PCA-Based Benign Flow Integration (CIC18 → CIC17)

This notebook evaluates the impact of integrating **benign network flows** from the target dataset (**CIC-IDS2018**) into the source dataset (**CIC-IDS2017**) before training a Machine Learning-based Intrusion Detection System (IDS).

To avoid data leakage, all flows integrated from the target dataset are removed from the target test set. In addition, an equivalent number of benign flows is removed from the source dataset before integration, preserving the overall dataset size and reducing distribution imbalance.

The dimensionality reduction step is performed with **Principal Component Analysis (PCA)**. PCA is fitted only on the training data after feature scaling, avoiding information leakage from the test and target datasets.

## 1. Import Libraries

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

## 2. Experiment Configuration

Adjust the paths below according to your local or repository directory structure.

In [ ]:
RANDOM_STATE = 42

SOURCE_DATASET_NAME = "CIC17"
TARGET_DATASET_NAME = "CIC18"

SOURCE_DATASET_PATH = Path(".../GenIDS-CIC17.csv")
TARGET_DATASET_PATH = Path(".../GenIDS-CIC18.csv")

LABEL_COLUMN = "binary"
BENIGN_LABEL = 0
MALICIOUS_LABEL = 1

INTEGRATION_RATES = [0.20, 0.40, 0.60, 0.80]

PCA_COMPONENTS = 25
TEST_SIZE = 0.80

DROP_COLUMNS_SOURCE = [
    "multiclass", "mapped_label", "date", "hours", "expiration_id",
    "src_ip", "src_mac", "src_oui", "dst_ip", "dst_mac", "dst_oui",
    "ip_version", "vlan_id", "tunnel_id"
]

DROP_COLUMNS_TARGET = [
    "multiclass", "label", "Timestamp",
    "src_ip", "src_mac", "src_oui", "dst_ip", "dst_mac", "dst_oui",
    "ip_version", "vlan_id", "tunnel_id"
]

CATEGORICAL_COLUMNS = ["application_name", "application_category_name", LABEL_COLUMN]
TIME_COLUMN = "bidirectional_first_seen_ms"

## 3. Helper Functions

In [ ]:
def load_dataset(file_path: Path) -> pd.DataFrame:
    """Load a CSV dataset from disk."""
    if not file_path.exists():
        raise FileNotFoundError(f"Dataset not found: {file_path}")
    return pd.read_csv(file_path)


def prepare_source_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare the source dataset by removing unused columns."""
    df = df.copy()
    columns_to_drop = [col for col in DROP_COLUMNS_SOURCE if col in df.columns]
    df.drop(columns=columns_to_drop, inplace=True)
    return df


def prepare_target_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare the target dataset by creating the binary label and removing unused columns."""
    df = df.copy()
    if LABEL_COLUMN not in df.columns and "label" in df.columns:
        df[LABEL_COLUMN] = df["label"].copy()
    elif "label" in df.columns:
        df[LABEL_COLUMN] = df["label"].copy()

    columns_to_drop = [col for col in DROP_COLUMNS_TARGET if col in df.columns]
    df.drop(columns=columns_to_drop, inplace=True)
    return df


def encode_categorical_columns(source_df: pd.DataFrame, target_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Encode categorical columns consistently across source and target datasets."""
    source_df = source_df.copy()
    target_df = target_df.copy()

    for column in CATEGORICAL_COLUMNS:
        if column in source_df.columns and column in target_df.columns:
            encoder = LabelEncoder()
            combined_values = pd.concat([source_df[column], target_df[column]], axis=0).astype(str)
            encoder.fit(combined_values)
            source_df[column] = encoder.transform(source_df[column].astype(str))
            target_df[column] = encoder.transform(target_df[column].astype(str))

    return source_df, target_df


def standardize_numeric_types(df: pd.DataFrame, label_column: str = LABEL_COLUMN) -> pd.DataFrame:
    """Standardize numeric columns to float64 and preserve the label column as int64."""
    df = df.copy()
    for column in df.columns:
        if column == label_column:
            df[column] = df[column].astype(np.int64)
        elif pd.api.types.is_numeric_dtype(df[column]):
            df[column] = df[column].astype(np.float64)
    return df


def print_class_distribution(df: pd.DataFrame, dataset_name: str, label_column: str = LABEL_COLUMN) -> None:
    """Print absolute and relative class distribution."""
    print(f"\n{dataset_name} - Absolute distribution")
    print(df[label_column].value_counts().sort_index())

    print(f"\n{dataset_name} - Relative distribution")
    print(df[label_column].value_counts(normalize=True).sort_index().map("{:.2%}".format))

In [ ]:
def integrate_benign_flows(
    source_df: pd.DataFrame,
    target_df: pd.DataFrame,
    integration_rate: float,
    label_column: str = LABEL_COLUMN,
    benign_label: int = BENIGN_LABEL,
    time_column: str = TIME_COLUMN,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Integrate benign flows from the target dataset into the source dataset.

    Steps:
    1. Select a percentage of benign flows from the target dataset.
    2. Remove selected target flows from the target test set to avoid data leakage.
    3. Remove the same number of benign flows from the source dataset.
    4. Concatenate the reduced source dataset with the selected target flows.
    5. Sort by flow timestamp when available.

    Returns:
        integrated_source_df: source dataset after target-flow integration
        target_test_df: target dataset after removing integrated flows
        selected_target_flows: target flows integrated into the source dataset
        removed_source_flows: source flows removed to preserve size
    """
    source_df = source_df.copy()
    target_df = target_df.copy()

    target_benign = target_df[target_df[label_column] == benign_label]
    number_of_flows_to_integrate = int(len(target_benign) * integration_rate)

    selected_target_flows = target_benign.iloc[:number_of_flows_to_integrate].copy()
    target_test_df = target_df.drop(index=selected_target_flows.index).copy()

    source_benign = source_df[source_df[label_column] == benign_label]
    number_of_source_flows_to_remove = min(number_of_flows_to_integrate, len(source_benign))

    removed_source_flows = source_benign.iloc[:number_of_source_flows_to_remove].copy()
    reduced_source_df = source_df.drop(index=removed_source_flows.index).copy()

    selected_target_flows["source_dataset"] = f"{TARGET_DATASET_NAME}_integrated_benign"
    reduced_source_df["source_dataset"] = SOURCE_DATASET_NAME

    integrated_source_df = pd.concat([reduced_source_df, selected_target_flows], ignore_index=True)

    if time_column in integrated_source_df.columns:
        integrated_source_df = integrated_source_df.sort_values(by=time_column).reset_index(drop=True)

    integrated_source_df.drop(columns=["source_dataset"], inplace=True)

    return integrated_source_df, target_test_df, selected_target_flows, removed_source_flows

In [ ]:
def split_features_and_labels(df: pd.DataFrame, label_column: str = LABEL_COLUMN) -> tuple[pd.DataFrame, pd.Series]:
    """Split a dataset into features and labels."""
    X = df.drop(columns=[label_column])
    y = df[label_column]
    return X, y


def build_xgboost_model(random_state: int = RANDOM_STATE) -> XGBClassifier:
    """Create the XGBoost classifier used in the experiments."""
    return XGBClassifier(
        eval_metric="logloss",
        n_estimators=300,
        max_depth=10,
        objective="binary:logistic",
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.5,
        random_state=random_state,
        n_jobs=-1,
    )


def compute_metrics(y_true, y_pred, y_pred_proba) -> dict:
    """Compute binary classification metrics."""
    cm = confusion_matrix(y_true, y_pred)

    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        far = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    else:
        far = np.nan

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "auc_roc": roc_auc_score(y_true, y_pred_proba),
        "auc_pr": average_precision_score(y_true, y_pred_proba, pos_label=MALICIOUS_LABEL),
        "far": far,
    }


def print_metrics(metrics: dict, title: str) -> None:
    """Print a formatted metrics report."""
    print(f"\n{title}")
    print("-" * len(title))
    for metric_name, metric_value in metrics.items():
        print(f"{metric_name}: {metric_value:.4f}")

In [ ]:
def plot_confusion_matrix(model, X, y, title: str) -> None:
    """Plot the confusion matrix."""
    ConfusionMatrixDisplay.from_estimator(model, X, y)
    plt.title(title)
    plt.show()


def plot_roc_curve(y_true, y_pred_proba, title: str) -> None:
    """Plot the ROC curve."""
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
    auc_roc = roc_auc_score(y_true, y_pred_proba)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f"AUC-ROC = {auc_roc:.4f}")
    plt.plot([0, 1], [0, 1], "k--", label="Random Guess")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()


def plot_precision_recall_curve(y_true, y_pred_proba, title: str) -> None:
    """Plot the Precision-Recall curve."""
    precision, recall, _ = precision_recall_curve(y_true, y_pred_proba, pos_label=MALICIOUS_LABEL)
    auc_pr = average_precision_score(y_true, y_pred_proba, pos_label=MALICIOUS_LABEL)
    baseline = np.sum(y_true) / len(y_true)

    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, label=f"AUC-PR = {auc_pr:.4f}")
    plt.plot([0, 1], [baseline, baseline], "k--", label="Baseline")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()


def plot_pca_explained_variance(pca: PCA) -> None:
    """Plot cumulative explained variance for PCA components."""
    cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

    plt.figure(figsize=(10, 6))
    plt.plot(cumulative_variance, marker="o")
    plt.xlabel("Principal Components")
    plt.ylabel("Cumulative Explained Variance")
    plt.title("Cumulative Explained Variance by PCA Components")
    plt.grid(True)
    plt.show()


def plot_pca_component_importance(model, top_n: int = 10) -> pd.DataFrame:
    """Plot and return the most important PCA components according to XGBoost."""
    importances = model.feature_importances_
    pca_features = [f"PC{i+1}" for i in range(len(importances))]

    importance_df = pd.DataFrame({
        "principal_component": pca_features,
        "importance": importances
    }).sort_values(by="importance", ascending=False)

    top_components = importance_df.head(top_n)

    plt.figure(figsize=(10, 6))
    plt.barh(top_components["principal_component"][::-1], top_components["importance"][::-1])
    plt.title(f"Top {top_n} Principal Components")
    plt.xlabel("Importance")
    plt.ylabel("Principal Component")
    plt.tight_layout()
    plt.show()

    return importance_df

## 4. Load and Prepare Datasets

In [ ]:
source_df_raw = load_dataset(SOURCE_DATASET_PATH)
target_df_raw = load_dataset(TARGET_DATASET_PATH)

source_df = prepare_source_dataset(source_df_raw)
target_df = prepare_target_dataset(target_df_raw)

source_df, target_df = encode_categorical_columns(source_df, target_df)

source_df = standardize_numeric_types(source_df)
target_df = standardize_numeric_types(target_df)

print(source_df.shape)
print(target_df.shape)

print_class_distribution(source_df, SOURCE_DATASET_NAME)
print_class_distribution(target_df, TARGET_DATASET_NAME)

## 5. Run PCA-Based Benign Flow Integration Experiments

The following loop evaluates all integration rates. For each rate:

1. Benign flows from the target dataset are selected for integration.
2. The selected flows are removed from the target test set to prevent data leakage.
3. The same number of benign flows is removed from the source dataset.
4. The selected target flows are integrated into the source dataset.
5. PCA is fitted only on the scaled training data.
6. XGBoost is trained and evaluated using intraset and interset scenarios.

In [ ]:
experiment_results = []

for integration_rate in INTEGRATION_RATES:
    print("=" * 100)
    print(f"Integration rate: {integration_rate:.0%}")
    print("=" * 100)

    integrated_source_df, target_test_df, integrated_target_flows, removed_source_flows = integrate_benign_flows(
        source_df=source_df,
        target_df=target_df,
        integration_rate=integration_rate,
    )

    print(f"Integrated target benign flows: {integrated_target_flows.shape}")
    print(f"Removed source benign flows: {removed_source_flows.shape}")
    print(f"Integrated source dataset: {integrated_source_df.shape}")
    print(f"Target test dataset after leakage-safe removal: {target_test_df.shape}")

    print_class_distribution(integrated_source_df, f"{SOURCE_DATASET_NAME} + {TARGET_DATASET_NAME} benign integration")
    print_class_distribution(target_test_df, f"{TARGET_DATASET_NAME} test set")

    X_source, y_source = split_features_and_labels(integrated_source_df)
    X_target, y_target = split_features_and_labels(target_test_df)

    X_train, X_intraset_test, y_train, y_intraset_test = train_test_split(
        X_source,
        y_source,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_source,
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_intraset_test_scaled = scaler.transform(X_intraset_test)
    X_target_scaled = scaler.transform(X_target)

    pca = PCA(n_components=PCA_COMPONENTS, random_state=RANDOM_STATE)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_intraset_test_pca = pca.transform(X_intraset_test_scaled)
    X_target_pca = pca.transform(X_target_scaled)

    print(f"Number of PCA components: {X_train_pca.shape[1]}")
    print(f"Total explained variance: {pca.explained_variance_ratio_.sum():.4f}")

    model = build_xgboost_model()
    model.fit(X_train_pca, y_train)

    # Intraset evaluation
    y_intraset_pred = model.predict(X_intraset_test_pca)
    y_intraset_proba = model.predict_proba(X_intraset_test_pca)[:, 1]
    intraset_metrics = compute_metrics(y_intraset_test, y_intraset_pred, y_intraset_proba)

    print_metrics(intraset_metrics, "Intraset Evaluation")
    print("\nIntraset Classification Report")
    print(classification_report(y_intraset_test, y_intraset_pred, digits=4, zero_division=0))

    # Interset evaluation
    y_interset_pred = model.predict(X_target_pca)
    y_interset_proba = model.predict_proba(X_target_pca)[:, 1]
    interset_metrics = compute_metrics(y_target, y_interset_pred, y_interset_proba)

    print_metrics(interset_metrics, "Interset Evaluation")
    print("\nInterset Classification Report")
    print(classification_report(y_target, y_interset_pred, digits=4, zero_division=0))

    experiment_results.append({
        "integration_rate": integration_rate,
        "integrated_target_benign_flows": len(integrated_target_flows),
        "removed_source_benign_flows": len(removed_source_flows),
        "source_train_size": len(X_train),
        "source_intraset_test_size": len(X_intraset_test),
        "target_test_size": len(X_target),
        "pca_components": PCA_COMPONENTS,
        "pca_explained_variance": pca.explained_variance_ratio_.sum(),
        **{f"intraset_{key}": value for key, value in intraset_metrics.items()},
        **{f"interset_{key}": value for key, value in interset_metrics.items()},
    })

## 6. Results Summary

In [ ]:
results_df = pd.DataFrame(experiment_results)
results_df

In [ ]:
results_output_path = Path("results_pca_benign_flow_integration.csv")
results_df.to_csv(results_output_path, index=False)
print(f"Results saved to: {results_output_path}")

## 7. Optional Diagnostic Plots

The cells below generate diagnostic plots for the last executed integration rate.
Run them after the experiment loop if visual inspection is required.

In [ ]:
plot_pca_explained_variance(pca)

In [ ]:
pca_importance_df = plot_pca_component_importance(model, top_n=10)
pca_importance_df.head(10)

In [ ]:
plot_confusion_matrix(
    model,
    X_intraset_test_pca,
    y_intraset_test,
    title="Intraset Confusion Matrix"
)

plot_roc_curve(
    y_intraset_test,
    y_intraset_proba,
    title="Intraset ROC Curve"
)

plot_precision_recall_curve(
    y_intraset_test,
    y_intraset_proba,
    title="Intraset Precision-Recall Curve"
)

In [ ]:
plot_confusion_matrix(
    model,
    X_target_pca,
    y_target,
    title="Interset Confusion Matrix"
)

plot_roc_curve(
    y_target,
    y_interset_proba,
    title="Interset ROC Curve"
)

plot_precision_recall_curve(
    y_target,
    y_interset_proba,
    title="Interset Precision-Recall Curve"
)